<a href="https://colab.research.google.com/github/Pablouski7/clustering_n_rag_u_index/blob/main/notebooks/eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Muestra de artículos de prensa (2019–2026) — EDA, clustering y embeddings

Pipeline completo sobre `data/raw/stratified_grid_2019_2026.csv` (~28,777 artículos de
**Diario Expreso**, **El Universo** y **Primicias**), en cinco secciones:

1. **EDA** — análisis exploratorio sobre el texto **crudo**: calidad, artefactos de
   PressReader, longitudes, cobertura año × periódico, secciones, serie temporal y flags.
2. **Data Wrangling** — todas las transformaciones: normalización por campo,
   deduplicación, **sección canónica** (label para el clustering) y descarte de artículos
   de ≤40 palabras.
3. **Embeddings** — tres métodos texto → vector sobre el corpus ya limpio, y su
   visualización en 3D con **PCA** y **UMAP**.
4. **Clustering** — **HDBSCAN** sobre cada método, con métricas internas y externas
   contra la sección canónica.
5. **AE / VAE** — reducción de dimensión sobre los embeddings (exploratorio).

Métodos de embedding:

| Método | Dims | Cómo cubre el artículo |
|---|---:|---|
| **Doc2Vec** (gensim, PV-DM) | 300 | bolsa de palabras, sin límite de contexto |
| **BGE-M3** vía vLLM (servidor H200) | 1024 | un solo forward (ventana 8,192 tokens) |
| **BETO** (`dccuchile/bert-base-spanish-wwm-cased`) | 768 | ventanas solapadas de 512 tokens + pooling |

**Nota de comparabilidad:** los tres ven el **artículo completo**, cada uno por su
mecanismo. Al igualar la cantidad de texto, cualquier diferencia observada apunta al modelo
y su mecanismo de cobertura, no a cuánto texto vio cada uno.


In [ ]:
import sys
from pathlib import Path

# Ancla la raíz del repo esté donde esté el kernel (local o Colab)
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Raíz del repo en el path (para importar src.llm_clients, como en scripts/check_vllm.py)
sys.path.insert(0, str(Path.cwd().parent))

# Paleta categórica fija por periódico (orden alfabético estable)
PALETA = ["#0073ff", "#00FF00", "#ff7b00"]  # azul, verde, magenta
COLOR_FLAG = {0: "#c3c2b7", 1: "#ff0000"}   # gris neutro / azul

sns.set_theme(style="whitegrid", palette=PALETA)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.color": "#e1e0d9",
                     "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e"})
RANDOM_STATE = 42

---
# 1. EDA

Análisis exploratorio **sin modificar los datos**: todo lo que sigue se mide sobre el texto
tal como llega del CSV. Las correcciones que este análisis justifica se aplican después, en
la sección 2 (Data Wrangling).


## Carga y tipos


In [ ]:
data = pd.read_csv(RAIZ / "data" / "raw" / "stratified_grid_2019_2026.csv")
data

In [ ]:
data.columns.to_list()

In [ ]:
data.drop(columns=["id_pressreader", "id_periodico", "icor_index", "icor_v2_index"], inplace=True)
data.rename({"incertidumbre": "u_index",
            "economic_uncertainty": "economic_u_index",
            "political_uncertainty": "political_u_index",},
            axis=1, inplace=True)

In [ ]:
data.info()

In [ ]:
data["fecha"] = pd.to_datetime(data["fecha"])
data["anio"] = pd.to_datetime(data["anio"], format="%Y").dt.year
data["fecha"], data["anio"]

## Calidad de datos

Nulos, duplicados y consistencia básica antes de cualquier análisis.

In [ ]:
print("Nulos por columna:")
print(data.isna().sum())
print(f"\nDuplicados por id_articulo: {data['id_articulo'].duplicated().sum()}")
print(f"Duplicados por (titulo, texto): {data.duplicated(subset=['titulo', 'texto']).sum()}")
print(f"Textos vacíos: {(data['texto'].str.strip().str.len() == 0).sum()}")
print(f"Rango de fechas: {data['fecha'].min().date()} → {data['fecha'].max().date()}")

## Análisis de calidad del texto

In [ ]:
import re
import unicodedata
import collections
import pandas as pd


def diagnosticar_textos(df, col="texto", n_ejemplos=3, ancho=160):
    """Audita la columna de texto: prevalencia de artefactos conocidos + descubrimiento
    de codepoints no-ASCII no esperados. Solo reporta, no modifica nada."""
    t = df[col].fillna("").astype(str)
    N = len(t)

    # --- Patrones conocidos: (nombre, regex, es_grave) ---
    PATRONES = [
        ("Palabras pegadas (min.Mayus)", r"[a-záéíóúñ][.,;:!?][A-ZÁÉÍÓÚÑ]", True),
        ("Soft hyphen \\xad (parte palabras)", r"\xad", True),
        ("Pictogramas de recuadro", r"[■⬛◼◗❚❑►▶●◉‣]", True),
        ("Crucigrama (HORIZONTALES/VERTICALES)", r"HORIZONTALES|VERTICALES", True),
        ("Comillas tipográficas", r"[“”‘’«»]", False),
        ("Guiones tipográficos", r"[–—−]", False),
        ("Código sección final (I)/(O)/(E)", r"\(\s*[IOEFR]\s*\)\s*$", False),
        ("URLs", r"https?://|www\.", False),
        ("Emails", r"\b[\w.-]+@[\w.-]+\.\w+", False),
        ("Acentos combinantes (no-NFC)", r"[\u0300-\u036f]", False),
        ("Viñetas numeradas ≥6 (listas)", None, False),   # caso especial: conteo
        ("Runs de -- __ ..", r"[-_.]{4,}", False),
        ("Marker recuadro AL INICIO", r"^\s*[■⬛◼◗❚❑►▶●]", True),
        ("Dígito pegado a letra (12años)", r"\d[a-záéíóúñ]{3,}|[a-záéíóúñ]{4,}\d{4}", False),
    ]

    print(f"{'='*70}\nDIAGNÓSTICO — columna '{col}', {N:,} textos\n{'='*70}")
    print(f"{'PATRÓN':<40}{'DOCS':>8}{'%':>8}  GRAVE")
    print("-" * 70)
    hallazgos = {}
    for nombre, patron, grave in PATRONES:
        if patron is None:  # viñetas numeradas: umbral por conteo
            mask = t.str.count(r"\b\d+\.\s") >= 6
        else:
            mask = t.str.contains(patron, regex=True)
        n = int(mask.sum())
        hallazgos[nombre] = mask
        marca = "  🔴" if (grave and n) else ""
        print(f"{nombre:<40}{n:>8,}{n/N*100:>7.2f}%{marca}")

    # --- Duplicados ---
    dup_texto = int(df.duplicated(subset=[col], keep=False).sum())
    print("-" * 70)
    print(f"{'Textos duplicados exactos':<40}{dup_texto:>8,}{dup_texto/N*100:>7.2f}%")

    # --- DESCUBRIMIENTO: codepoints no-ASCII no esperados ---
    ESPERADOS = set("áéíóúñüÁÉÍÓÚÑÜ¿¡«»“”‘’–—−…■⬛◼◗❚❑►▶●")
    ESPERADOS |= set("çèãöëàäïô×øêâåºª°´²³©®™°")  # acentos/símbolos ya vistos
    cont = collections.Counter()
    for s in t:
        for ch in s:
            if ord(ch) > 127 and ch not in ESPERADOS and not unicodedata.combining(ch):
                cont[ch] += 1
    if cont:
        print(f"\n{'CODEPOINTS NO-ASCII INESPERADOS (posibles hallazgos nuevos)':^70}")
        print(f"{'CODEPOINT':<12}{'CAT':<5}{'OCURR':>8}{'DOCS':>7}  NOMBRE / ejemplo")
        print("-" * 70)
        for ch, oc in cont.most_common(25):
            docs = int(t.str.contains(re.escape(ch)).sum())
            nombre_u = unicodedata.name(ch, "?")
            print(f"U+{ord(ch):04X}{'':<6}{unicodedata.category(ch):<5}{oc:>8,}{docs:>7}  {nombre_u} {ch!r}")

    # --- Ejemplos concretos de los patrones graves ---
    print(f"\n{'='*70}\nEJEMPLOS (patrones graves)\n{'='*70}")
    for nombre, mask in hallazgos.items():
        graves = {p[0] for p in PATRONES if p[2]}
        if nombre not in graves or mask.sum() == 0:
            continue
        print(f"\n### {nombre} ({int(mask.sum()):,} docs)")
        for s in t[mask].head(n_ejemplos):
            frag = re.sub(r"\s+", " ", s).strip()
            print(f"  · {frag}")

    return hallazgos  # dict {patrón: máscara booleana} por si quieres filtrar con ellos

In [ ]:
# El diagnóstico se ejecuta sobre titulo + seccion + texto: los artefactos no viven solo
# en el cuerpo (las etiquetas de sección casi duplican la cuenta de soft hyphens). El
# separador "\n" no crea falsos "palabras pegadas" en los bordes entre campos.
data["texto_diag"] = (data["titulo"].fillna("") + "\n"
                      + data["seccion"].fillna("") + "\n"
                      + data["texto"].fillna(""))
hallazgos = diagnosticar_textos(data, col="texto_diag")

### Diagnóstico de artefactos de texto (PressReader)

El texto proviene de PressReader y arrastra artefactos de maquetación que
**no son contenido** y degradan silenciosamente los embeddings (sobre todo en modelos
subword como BETO y BGE-M3, donde un carácter invisible a media palabra rompe la
tokenización). El diagnóstico se ejecuta sobre la concatenación `titulo + seccion + texto`,
para cubrir todo el material que llega al modelo (título y cuerpo se embeben juntos) más la
etiqueta de sección. Prevalencia medida sobre el corpus completo (28,777 artículos):

| Artefacto | Docs | % | Impacto |
|---|---:|---:|---|
| Palabras pegadas tras puntuación (`Lloret.En`) | 13,611 | 47.3% | Alto |
| Comillas tipográficas (`" " ' '`) | 12,325 | 42.8% | Bajo (cosmético) |
| Código de sección final `(I)`/`(O)`/`(E)` | 3,293 | 11.4% | Bajo |
| Soft hyphen `\xad` (parte palabras: `Actualidad`) | 2,359 | 8.2% | Alto |
| Guiones tipográficos (`– — −`) | 1,361 | 4.7% | Bajo |
| Pictogramas de recuadro (`■ ▶ ⬛ ◗ • ❏ ▪`) | 851 | 3.0% | Medio (ruido puro) |
| Dígito pegado a letra (`12años`) | 142 | 0.5% | Bajo |
| Crucigramas (`HORIZONTALES`/`VERTICALES`) | 32 | 0.1% | Alto (contenido no temático) |
| Private Use Area `\uF07D`, non-breaking hyphen `\u2011`, variation selectors `\uFE0F` | <10 c/u | — | Bajo |

**Observaciones clave:**

- **Los artefactos no viven solo en `texto`.** Al añadir `titulo` y `seccion` al diagnóstico,
  el soft hyphen casi se duplica (1,345 → 2,359 docs): las **etiquetas de sección vienen
  partidas** (`Actualidad`, `Diversión`). Esto no es cosmético: `seccion` alimenta la
  canonicalización de secciones, y `Actualidad` ≠ `Actualidad` haría
  cuente como dos. Por eso `seccion` también necesita al menos la limpieza de invisibles.
- El artefacto dominante son las **palabras pegadas** (47%): PressReader pierde el espacio tras
  el punto final de oración. Es corregible **solo** cuando queda punt
  palabras; la fracción pegada sin puntuación (`deImpuesto`, `mientraspermanecía`) es
  **irrecuperable** sin un segmentador y se delega en la tokenización subword del modelo.
- El **soft hyphen** (`\xad`, 95k ocurrencias) es el más traicionero:
  el tokenizer lo procesa y parte `primer` en subtokens espurios.
- Se detectó un carácter **Private Use Area** (`U+F07D`, categoría `C
  de la fuente de PressReader, sin significado Unicode → basura que se elimina.
- Los caracteres extranjeros detectados (`č ć š ž Ö Ç à ½ ₂`) son **contenido legítimo**
  (nombres eslavos/germánicos, fracciones, fórmulas) y **no** se tocan. No hay duplicados de
  texto (0 exactos).

## Longitud de los textos

In [ ]:
# Longitudes sobre el texto CRUDO (aún sin normalizar). La sección 2 vuelve a medirlas
# tras la limpieza, y la comparación antes/después es en sí un resultado.
data["n_caracteres"] = data["texto"].str.len()
data["n_palabras"] = data["texto"].str.split().str.len()
data[["texto", "n_caracteres", "n_palabras"]].head(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

bins_palabras = np.logspace(np.log10(data["n_palabras"].clip(lower=1).min()),
                            np.log10(data["n_palabras"].max()), 60)
axes[0].hist(data["n_palabras"], bins=bins_palabras, color="#2a78d6", edgecolor="white", linewidth=0.3)
axes[0].set_xscale("log")
axes[0].set_xlabel("Palabras por artículo (log)")
axes[0].set_ylabel("Artículos")
axes[0].set_title("Distribución de longitud (palabras)")

bins_caracteres = np.logspace(np.log10(data["n_caracteres"].clip(lower=1).min()),
                              np.log10(data["n_caracteres"].max()), 60)
axes[1].hist(data["n_caracteres"], bins=bins_caracteres, color="#2a78d6", edgecolor="white", linewidth=0.3)
axes[1].set_xscale("log")
axes[1].set_xlabel("Caracteres por artículo (log)")
axes[1].set_title("Distribución de longitud (caracteres)")

orden_per = sorted(data["nombre_periodico"].unique())
sns.boxplot(data=data, x="nombre_periodico", y="n_palabras", order=orden_per,
            hue="nombre_periodico", hue_order=orden_per, palette=PALETA,
            legend=False, ax=axes[2], fliersize=1, linewidth=1)
axes[2].set_yscale("log")
axes[2].set_xlabel("")
axes[2].set_ylabel("Palabras por artículo (log)")
axes[2].set_title("Longitud por periódico")

plt.tight_layout()
plt.show()

data[["n_palabras", "n_caracteres"]].describe().round(0)

In [ ]:
# Outliers en n_palabras según criterio IQR (1.5 * IQR)
q1 = data["n_palabras"].quantile(0.25)
q3 = data["n_palabras"].quantile(0.75)
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

print(f"Q1={q1:.0f}  Q3={q3:.0f}  IQR={iqr:.0f}")
print(f"Límites 'normales': [{limite_inferior:.0f}, {limite_superior:.0f}]")

outliers = data[(data["n_palabras"] < limite_inferior) | (data["n_palabras"] > limite_superior)]
print(f"\nTotal outliers: {len(outliers)} ({len(outliers)/len(data)*100:.1f}%)")
print(f"  - por debajo (< {limite_inferior:.0f}): {(data['n_palabras'] < limite_inferior).sum()}")
print(f"  - por encima (> {limite_superior:.0f}): {(data['n_palabras'] > limite_superior).sum()}")

# Los más extremos por arriba (artículos larguísimos)
cols = [c for c in ["id_articulo", "titulo", "nombre_periodico", "seccion", "n_palabras", "n_caracteres"] if c in data.columns]
outliers.sort_values("n_palabras", ascending=False)[cols]


In [ ]:
# Distribución fina de la cola baja: dimensiona cuántos artículos son teletipos, pies de
# foto o restos de maquetación. Motiva el umbral de descarte que se aplica en la sección 2.
for umbral in [5, 10, 20, 30, 40, 50, 60]:
    n = (data['n_palabras'] <= umbral).sum()
    print(f"<= {umbral:>3} palabras: {n:>5} artículos ({n/len(data)*100:.2f}%)")

print("\n--- Ejemplos de los más cortos ---")
cortos = data[data['n_palabras'] <= 50].sort_values('n_palabras').tail()
print(cortos[['n_palabras', 'texto']].to_string(index=False))

## Cobertura muestral: año × periódico

Verificación del diseño estratificado (cuotas por estrato año × periódico).

In [ ]:
cobertura = data.pivot_table(index="nombre_periodico", columns="anio",
                             values="id_articulo", aggfunc="count", fill_value=0)

fig, ax = plt.subplots(figsize=(10, 2.8))
sns.heatmap(cobertura, annot=True, fmt="d", cmap="Blues", linewidths=2,
            linecolor="white", cbar_kws={"label": "Artículos"}, ax=ax)
ax.set_xlabel("Año")
ax.set_ylabel("")
ax.set_title("Artículos por estrato (año × periódico)")
plt.tight_layout()
plt.show()

In [ ]:
orden_per = data["nombre_periodico"].value_counts().index

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=data, y="nombre_periodico", order=orden_per, color=PALETA[0], ax=ax)
ax.set_xlabel("Cantidad de artículos")
ax.set_ylabel("")
ax.set_title("Artículos por periódico")

for cont in ax.containers:
    ax.bar_label(cont, padding=3)

sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()


## Secciones (crudas)

La sección viene como la titula cada periódico, así que es de **altísima cardinalidad**:
hay cientos de etiquetas distintas y la mayoría aparece en uno o dos artículos. Esto es
justo lo que motiva el mapeo a secciones canónicas de la sección 2 — como label de
clustering, la sección cruda es inservible.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=False)
for ax, periodico, color in zip(axes, orden_per, PALETA):
    top = (data.loc[data["nombre_periodico"] == periodico, "seccion"]
           .value_counts().head(20).sort_values())
    ax.barh(top.index, top.values, color=color, height=0.7)
    ax.set_title(periodico)
    ax.set_xlabel("Artículos")
    ax.tick_params(labelsize=8)
plt.suptitle("Top 10 secciones por periódico", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Por periódico: nº de secciones que aparecen una sola vez ("únicas") frente al total
# de secciones distintas, para dimensionar la cola de etiquetas de un solo artículo.
resumen = []
for periodico in orden_per:
    vc = data.loc[data["nombre_periodico"] == periodico, "seccion"].value_counts()
    resumen.append({
        "periodico": periodico,
        "secciones_totales": vc.size,
        "secciones_unicas": int((vc == 1).sum()),
        "%_unicas": round(100 * (vc == 1).mean(), 1),
    })

resumen = pd.DataFrame(resumen).set_index("periodico")
resumen


## Serie temporal: artículos por mes

In [ ]:
mensual = (data.set_index("fecha").groupby("nombre_periodico")
           .resample("ME")["id_articulo"].count().rename("articulos").reset_index())

fig, ax = plt.subplots(figsize=(12, 4))
for periodico, color in zip(orden_per, PALETA):
    serie = mensual[mensual["nombre_periodico"] == periodico]
    ax.plot(serie["fecha"], serie["articulos"], color=color, linewidth=2, label=periodico)
ax.set_xlabel("Mes")
ax.set_ylabel("Artículos")
ax.set_title("Artículos muestreados por mes y periódico")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## Flags de incertidumbre y corrupción

- `u_index`, `economic_u_index`, `political_u_index`: flags de incertidumbre ya
  almacenadas en la BD de origen.
- `icor_v3_1_index`: flag de detección de corrupción (versión 3.1, con exclusión).

In [ ]:
FLAGS = ["u_index", "economic_u_index", "political_u_index", "icor_v3_1_index"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

prevalencia = data[FLAGS].mean().sort_values() * 100
axes[0].barh(prevalencia.index, prevalencia.values, color="#2a78d6", height=0.6)
for i, v in enumerate(prevalencia.values):
    axes[0].text(v + 0.3, i, f"{v:.1f}%", va="center", fontsize=9, color="#52514e")
axes[0].set_xlabel("% de artículos con flag = 1")
axes[0].set_title("Prevalencia global de cada flag")

anual = data.groupby("anio")[FLAGS].mean() * 100
colores_flags = ["#2a78d6", "#008300", "#e87ba4", "#eda100"]
for flag, color in zip(FLAGS, colores_flags):
    axes[1].plot(anual.index, anual[flag], color=color, linewidth=2, marker="o",
                 markersize=5, label=flag)
axes[1].set_xlabel("Año")
axes[1].set_ylabel("% de artículos")
axes[1].set_title("Prevalencia anual de las flags")
axes[1].legend(frameon=False, fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

por_periodico = data.groupby("nombre_periodico")[FLAGS].mean().loc[orden_per] * 100
por_periodico.plot(kind="bar", ax=axes[0], color=colores_flags, width=0.8, edgecolor="white")
axes[0].set_xlabel("")
axes[0].set_ylabel("% de artículos")
axes[0].set_title("Prevalencia de flags por periódico")
axes[0].legend(frameon=False, fontsize=9)
axes[0].tick_params(axis="x", rotation=0)

sns.heatmap(data[FLAGS].corr(), annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
            linewidths=2, linecolor="white", ax=axes[1], cbar_kws={"label": "Correlación"})
axes[1].set_title("Co-ocurrencia entre flags (correlación)")

plt.tight_layout()
plt.show()

---
# 2. Data Wrangling

Aquí se aplican, en orden, todas las transformaciones que el EDA justificó:

1. **Normalización por campo** (limpieza de carácter).
2. **Deduplicación** sobre el texto normalizado.
3. **Sección canónica**: colapsa las secciones crudas a ~17 macro-secciones, que serán el
   label de referencia del clustering.
4. **Descarte de artículos de ≤40 palabras**.

A partir de aquí `data` **sí** se modifica; el EDA de arriba ya no es reproducible sobre el
`data` resultante.


## Normalización de texto

### Diseño de la normalización

La normalización se restringe a **limpieza de carácter**; los descartes por contenido
(crucigramas, artículos demasiado cortos) se hacen aparte, por fila. Se aplica **por campo**,
porque no todos alimentan lo mismo:

- `titulo` y `texto` se **embeben** (entran como `titulo + ". " + texto` al modelo) → reciben
  la normalización completa (`norm_text`).
- `seccion` es una **etiqueta categórica** que alimenta la canonicalización, no el modelo →
  recibe una versión ligera (`norm_label`): quita invisibles y recompone Unicode, pero **no**
  repara palabras pegadas ni pictogramas (irrelevante en una etiqueta), para no alterar el
  valor que se agrupa.

`norm_text` ejecuta, en orden deliberado (cada paso asume el anterior):

1. **Borrar caracteres invisibles/sin sentido:** soft hyphen (`\xad`), variation selectors
   (`\uFE0E\uFE0F`), zero-width y **Private Use Area** (`\uE000–\uF8FF`). Se eliminan (no se
   reemplazan por espacio) para volver a unir la palabra partida (`Actualidad → Actualidad`).
2. **Non-breaking hyphen (`\u2011`) → guión normal (`-`).**
3. **Normalización Unicode `NFC`** (no `NFKC`): recompone acentos descompuestos sin alterar
   comillas, `ñ`, ordinales ni cifras. `NFKC` se descartó porque convertiría `²`, `º`,
   ligaduras y fracciones, alterando datos numéricos legítimos sin beneficio aquí.
4. **Pictogramas de maquetación → espacio.** El `■`/`▶` aparecen también a media frase
   (`CULTURA■ Está preso`), así que se sustituyen por espacio, no se borran, para no volver a
   pegar palabras.
5. **Reparar palabras pegadas: `min.Mayus → min. Mayus`.** El patrón exige
   *minúscula + puntuación + Mayúscula*, lo que evita romper siglas (`FF.AA`), decimales
   (`4.759`) y dominios (`.com`). Es el paso con mayor superficie de error, por eso se acota al
   caso inequívoco.
6. **Minúsculas y colapso de espacios** (`\s+ → " "`), al final, para absorber los espacios
   que introdujeron los pasos 4 y 5.

**Lo que NO se normaliza (por diseño):** comillas y guiones tipográficos (cosmético para
tokenizers subword), el código `(I)/(O)` final (inofensivo tras minúsculas) y las palabras
pegadas sin puntuación (irrecuperables). Tras aplicar la función, re-ejecutar el diagnóstico
sobre el texto normalizado confirma que los invisibles y pictogramas caen a ~0; el residuo
esperado son esos tres casos.

In [ ]:
import re
import unicodedata

# Pictogramas / viñetas de maquetación PressReader
_BASURA_PICTO = re.compile(r"[■⬛◼◗❚❑►▶▪●◉•‣·❏⬇◄▲▼★☆➤—―]")
# Invisibles / sin sentido: variation selectors, zero-width, direccionales y Private Use Area
_INVISIBLES = re.compile(r"[\uFE0E\uFE0F\u200B-\u200F\u202A-\u202E\uE000-\uF8FF]")


def _limpiar_invisibles(text: str) -> str:
    """Paso común: quita caracteres invisibles/sin sentido y recompone Unicode (NFC)."""
    text = text.replace("\xad", "")           # soft hyphen -> une palabra partida
    text = text.replace("\u2011", "-")        # non-breaking hyphen -> guión normal
    text = _INVISIBLES.sub("", text)          # variation selectors, zero-width, PUA (\uF07D)
    return unicodedata.normalize("NFC", text) # acentos descompuestos; NO altera comillas/ñ


def norm_text(text: str) -> str:
    """Normalización completa para contenido que se embebe (titulo, texto)."""
    text = _limpiar_invisibles(text)
    text = _BASURA_PICTO.sub(" ", text)                                      # pictogramas -> espacio
    text = re.sub(r"([a-záéíóúñ])([.,;:!?])([A-ZÁÉÍÓÚÑ])", r"\1\2 \3", text)  # min.Mayus -> min. Mayus
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def norm_label(text: str) -> str:
    """Normalización ligera para etiquetas categóricas (seccion): sin reparar pegados/pictos."""
    text = _limpiar_invisibles(text)
    text = _BASURA_PICTO.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()


# --- Aplicación por campo ---
data["titulo_norm"]  = data["titulo"].fillna("").map(norm_text)
data["texto_norm"]   = data["texto"].map(norm_text)
data["seccion"]      = data["seccion"].fillna("").map(norm_label)  # antes de seccion_canonica


data["texto_diag"] = (data["titulo_norm"] + "\n" + data["seccion"] + "\n" + data["texto_norm"])
_ = diagnosticar_textos(data, col="texto_diag")

In [ ]:
print("Duplicados tras normalizar:", int(data.duplicated(subset=["titulo_norm","texto_norm"]).sum()))

In [ ]:
data = data.drop_duplicates(subset=["titulo_norm","texto_norm"]).reset_index(drop=True)

In [ ]:
data.drop(columns=["texto_diag"], inplace=True)

## Sección canónica

La sección cruda tiene ~1,700 valores distintos en esta muestra (~7,400 en la BD completa),
la mayoría con 1-3 artículos: no se puede usar como clase ni balancear. El mapeo canónico
la colapsa a **17 macro-secciones** mediante reglas por palabra clave sobre el nombre
normalizado, evaluadas en orden (la primera que matchea gana).

La lógica es una copia literal de
`scripts/sampling_for_clustering/sample_balanced_by_seccion.py` (`normalizar_seccion`,
`_REGLAS`, `mapear_seccion_canonica`). Se **copia y no se importa** porque ese script
depende de `config.get_sqlalchemy_url` y `src.utilities.text_utils`, módulos que no existen
en este repositorio.

**Por qué recalcularla aquí si el CSV ya la trae:** la columna `seccion_canonica` del CSV
viene precocinada por el script de muestreo, contra la sección **cruda** de la BD. Traer la
lógica al notebook la vuelve **auditable y afinable** — sin esto, el ~6-7% que cae en
`Otros` no se puede mejorar sin volver a la base de datos. La celda de verificación mide
si recalcularla sobre la sección ya normalizada cambia algo (no debería: `normalizar_seccion`
aplica NFKC + minúsculas y matchea por substring, así que ya absorbe soft hyphens y
pictogramas por su cuenta).

Nota: `Portada`, `Opinión` y `Actualidad` no son temas sino formato o posición en el
periódico, y `Otros` es un cajón de sastre. La sección 4 reporta las métricas con y sin
estas cuatro clases.


In [ ]:
# --- Mapeo canónico de sección ---
# Copia literal de scripts/sampling_for_clustering/sample_balanced_by_seccion.py
# (normalizar_seccion, _REGLAS, mapear_seccion_canonica). Ver nota arriba sobre por qué se
# copia en vez de importarse.
import re
import unicodedata


def normalizar_seccion(s: str) -> str:
    """Normaliza el nombre de sección para el matching de reglas.

    Quita soft-hyphens, aplica NFKC, unifica separadores (& -> ' y ', '-' -> ' '),
    colapsa espacios y pasa a minúsculas.
    """
    if not isinstance(s, str):
        return ""
    s = s.replace("\xad", "")  # soft-hyphen
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("&", " y ").replace("-", " ")
    return re.sub(r"\s+", " ", s).strip().lower()


# Reglas por palabra clave (substring sobre el nombre normalizado). Orden = prioridad:
# la primera que matchea gana. Cubren la cabeza y la cola larga.
_REGLAS = [
    ("Deportes", ("deporte", "jugada", "marcador", "futbol", "fútbol",
                  "eliminatoria", "mundial", "olimp", "diversión", "diversion",
                  "juegos", "liga", "copa")),
    ("Economía", ("económ", "economia", "economía", "negocio", "entorno económico",
                  "enfoque económico", "mercado", "finanz")),
    ("Política", ("polít", "politic", "legislat", "elecc", "asamblea",
                  "coyuntura nacional", "acontecer nacional", "hechos del país",
                  "debate", "gobierno", "presidencial")),
    ("Seguridad", ("segurid", "suceso", "delict", "violent", "muerte", "crimen",
                   "expediente", "narco", "conflicto")),
    ("Ciencia y Tecnología", ("ciencia", "tecnolog", "tecno", "digital", "innovac")),
    ("Vida y Estilo", ("vida y estilo", "vida estilo", "vidayestilo", "estilo",
                       "gastronom", "moda", "salud", "bienestar", "en ruta",
                       "hogar", "familia")),
    ("Cultura", ("cultura", "música", "musica", "cine", "arte", "libro",
                 "literatura", "gente", "patrimonio")),
    ("Entretenimiento", ("entreten", "farándula", "farandula", "trending",
                         "qué ver", "que ver", "espectác", "espectac", "tv",
                         "televisión", "viral")),
    ("Mundo", ("mundo", "internacional", "panorama internacional", "el país",
               "el mundo", "global", "migra")),
    ("Local Guayaquil", ("guayaquil", "guayas", "gran guayaquil")),
    ("Local Quito", ("quito", "capital", "los valles", "pichincha")),
    ("Educación", ("educ", "universidad", "escolar")),
    ("Sociedad", ("sociedad", "comunidad", "intercultural", "información general",
                  "informacion general", "ecología", "ecologia", "ambiente",
                  "en la ciudad")),
    # No temáticas (se evalúan al final para no capturar prefijos temáticos).
    ("Portada", ("portada",)),
    ("Opinión", ("opinión", "opinion", "lectores", "editorial", "columna")),
    ("Actualidad", ("actualidad", "lo último", "lo ultimo", "hoy", "última hora")),
]


def mapear_seccion_canonica(seccion: str) -> str:
    """Mapea una sección cruda a su sección canónica (o 'Otros')."""
    norm = normalizar_seccion(seccion)
    if not norm:
        return "Otros"
    for canonica, claves in _REGLAS:
        if any(clave in norm for clave in claves):
            return canonica
    return "Otros"

In [ ]:
# Verificación: ¿recalcular la canónica sobre la sección ya normalizada cambia algo frente
# a la columna que trae el CSV? Se mide ANTES de sobrescribirla, para no perder la referencia.
canonica_regen = data["seccion"].map(mapear_seccion_canonica)
difieren = canonica_regen != data["seccion_canonica"]

print(f"Filas que cambian de clase al regenerar: {difieren.sum()} "
      f"({difieren.mean() * 100:.3f}%)")
if difieren.any():
    print("\nCasos que cambian (seccion -> del CSV / regenerada):")
    print(data.loc[difieren, ["seccion", "seccion_canonica"]]
          .assign(regenerada=canonica_regen[difieren])
          .value_counts().head(15).to_string())

data["seccion_canonica"] = canonica_regen

In [ ]:
# Auditoría del mapeo: distribución por clase, cobertura fuera de 'Otros' y las secciones
# crudas que más caen en 'Otros' (es la lista para afinar _REGLAS sin salir del notebook).
conteos = data["seccion_canonica"].value_counts()
n_otros = conteos.get("Otros", 0)

print("Distribución por sección canónica:")
print(conteos.to_string())
print(f"\nClases: {conteos.size}  |  cobertura fuera de 'Otros': "
      f"{(len(data) - n_otros) / len(data) * 100:.1f}% "
      f"({len(data) - n_otros:,} de {len(data):,})")

otros_crudos = data.loc[data["seccion_canonica"] == "Otros", "seccion"].value_counts()
print(f"\nTop secciones crudas que caen en 'Otros' ({otros_crudos.size} distintas):")
print(otros_crudos.head(25).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=False)
for ax, periodico, color in zip(axes, orden_per, PALETA):
    top = (data.loc[data["nombre_periodico"] == periodico, "seccion_canonica"]
           .value_counts().sort_values())
    ax.barh(top.index, top.values, color=color, height=0.7)
    ax.set_title(periodico)
    ax.set_xlabel("Artículos")
    ax.tick_params(labelsize=8)
plt.suptitle("Top secciones canonicas por periódico", y=1.02)
plt.tight_layout()
plt.show()

### La sección canónica como label del clustering

Estas 17 clases son la **referencia externa** con la que se evalúa el clustering en la
sección 4. Conviene ser explícito sobre qué son y qué no:

**Por qué sirven:**

- Es la **única etiqueta temática disponible** en el corpus. La alternativa, `seccion`
  cruda, tiene ~1,700 valores distintos y no se puede usar como clase.
- Tras el colapso quedan clases de **volumen comparable** (del orden de cientos a un par de
  miles de artículos cada una), porque `sample_grid_balanced.py` ya aplanó la sección
  dentro de cada celda año × periódico. Sin ese aplanado, el desbalance en el pool crudo
  llega a ~58×, y cualquier métrica supervisada estaría dominada por dos o tres clases.
- Permiten métricas **externas** (ARI, AMI, homogeneidad, completitud) que responden a la
  pregunta que importa: *¿la estructura que encuentra el clustering no supervisado se
  parece a la organización temática que hacen los periódicos?*

**Por qué NO son ground truth — y hay que decirlo al interpretar:**

- La etiqueta la asigna el periódico por criterio **editorial y de maquetación**, no por
  contenido: un mismo hecho puede ir en «Actualidad», en «Portada» o en la sección local
  según el día y el medio.
- El mapeo canónico es por **substring** sobre el nombre de página, así que hereda los
  errores de ese nombre y añade los suyos (una página «Mundial de fútbol» cae en Deportes,
  pero «Mundo» y «mundial» comparten prefijo con reglas distintas — de ahí que el orden de
  `_REGLAS` sea parte de la definición).
- Cuatro clases no son temas: `Portada` y `Actualidad` son posición o inmediatez,
  `Opinión` es género, `Otros` es residuo. Mezclan contenido heterogéneo por construcción.

**Consecuencia práctica:** un ARI bajo es ambiguo. Puede significar que el clustering
falló, o que encontró una partición temática **mejor** que la editorial. Por eso la sección
4 reporta las métricas en dos vistas —las 17 clases y solo las 13 temáticas— y acompaña
siempre las métricas externas con las internas y con el % de ruido.


## Descarte de artículos demasiado cortos

El EDA mostró una cola baja de artículos de pocas palabras: teletipos, pies de foto,
sumarios y restos de maquetación que no son artículos. Para embeddings y clustering son
ruido puro —un texto de 15 palabras produce un vector dominado por dos o tres términos— así
que se descartan los de **≤40 palabras**.

El umbral es una decisión de corte: 40 palabras es aproximadamente un titular más una
entradilla, el mínimo para que un embedding capture algo de tema. Se verifica abajo que el
descarte no se concentre en una sección o un periódico, lo que sesgaría el corpus.


In [ ]:
# Longitudes sobre el texto YA normalizado (las del EDA eran sobre el crudo).
data["n_caracteres"] = data["texto_norm"].str.len()
data["n_palabras"] = data["texto_norm"].str.split().str.len()

MIN_PALABRAS = 40
descartados = data[data["n_palabras"] <= MIN_PALABRAS]
n_antes = len(data)

print(f"Artículos con <= {MIN_PALABRAS} palabras: {len(descartados):,} "
      f"({len(descartados) / n_antes * 100:.2f}%)")

# ¿El descarte sesga alguna clase? Se compara la tasa de descarte por grupo con la global.
tasa_global = len(descartados) / n_antes
for col in ["nombre_periodico", "seccion_canonica"]:
    tasa = (data.groupby(col)["n_palabras"].apply(lambda s: (s <= MIN_PALABRAS).mean())
            .sort_values(ascending=False) * 100)
    print(f"\n% descartado por {col} (global: {tasa_global * 100:.2f}%):")
    print(tasa.round(2).to_string())

print("\n--- Ejemplos de artículos descartados ---")
print(descartados[["n_palabras", "titulo", "texto_norm"]]
      .sort_values("n_palabras", ascending=False).head(5).to_string(index=False, max_colwidth=90))

In [ ]:
# `reset_index(drop=True)`: la sección 4 indexa con `data.iloc[i]` a partir de posiciones
# de los embeddings, así que el índice debe quedar 0..N-1 sin huecos.
data = data[data["n_palabras"] > MIN_PALABRAS].reset_index(drop=True)

print(f"Corpus limpio: {len(data):,} artículos (de {n_antes:,} antes del filtro)")
print(f"Secciones canónicas: {data['seccion_canonica'].nunique()} "
      f"| nulos: {int(data['seccion_canonica'].isna().sum())}")
print(f"\nLongitud tras la limpieza:")
print(data[["n_palabras", "n_caracteres"]].describe().round(0).to_string())

---
# 3. Embeddings

Los tres métodos se calculan sobre el **corpus completo ya limpio** y se cachean en
`data/embeddings/` junto con los `id_articulo`: si los ids coinciden, se reutiliza el caché
en vez de recalcular. Cambiar el corpus (otro CSV, otro umbral de palabras) invalida los
cachés automáticamente.

⚠️ **Los checkpoints parciales no se autoinvalidan.** BGE-M3 y BETO guardan un
`*_full_parcial.npy` para poder reanudar si se corta la ejecución. Ese parcial **no guarda
ids**: si cambias de corpus y queda uno de una corrida anterior, la reanudación lo dará por
bueno mientras tenga menos filas que el corpus nuevo, y los vectores quedarán desalineados
sin ningún error visible. Al cambiar de corpus, **borra los parciales a mano**:

```bash
rm -f data/embeddings/*_full_parcial.npy
```


In [ ]:
# Entrada de los tres métodos: titulo + texto normalizados, completo y sin truncar.
textos = (data["titulo_norm"] + ". " + data["texto_norm"]).tolist()
print(f"{len(textos):,} textos | palabras: mín {data['n_palabras'].min()}, "
      f"mediana {data['n_palabras'].median():.0f}, máx {data['n_palabras'].max()}")
print(f"\nEjemplo:\n{textos[0][:300]}…")

In [ ]:
from pathlib import Path

EMB_DIR = RAIZ / "data" / "embeddings"
EMB_DIR.mkdir(parents=True, exist_ok=True)
ids_full = data["id_articulo"].to_numpy()


def cargar_cache_full(nombre: str) -> np.ndarray | None:
    """Devuelve el embedding cacheado si corresponde exactamente al corpus actual."""
    ruta_emb = EMB_DIR / f"{nombre}_full.npy"
    ruta_ids = EMB_DIR / f"{nombre}_full_ids.npy"
    if ruta_emb.exists() and ruta_ids.exists() and np.array_equal(np.load(ruta_ids), ids_full):
        print(f"Caché válido: {ruta_emb}")
        return np.load(ruta_emb)
    return None


def guardar_cache_full(nombre: str, emb: np.ndarray) -> None:
    np.save(EMB_DIR / f"{nombre}_full.npy", emb)
    np.save(EMB_DIR / f"{nombre}_full_ids.npy", ids_full)

## Doc2Vec

In [ ]:
emb_d2v = cargar_cache_full("doc2vec")
if emb_d2v is None:
    from gensim.models.doc2vec import Doc2Vec, TaggedDocument

    corpus_d2v = [TaggedDocument(words=texto.split(), tags=[i])
                  for i, texto in enumerate(textos)]
    modelo_d2v = Doc2Vec(vector_size=300, dm=1, window=8, min_count=3, epochs=40,
                         workers=4, seed=RANDOM_STATE)
    modelo_d2v.build_vocab(corpus_d2v)
    modelo_d2v.train(corpus_d2v, total_examples=modelo_d2v.corpus_count,
                     epochs=modelo_d2v.epochs)
    emb_d2v = np.array([modelo_d2v.dv[i] for i in range(len(corpus_d2v))],
                       dtype=np.float32)
    guardar_cache_full("doc2vec", emb_d2v)
print(f"Embeddings Doc2Vec: {emb_d2v.shape}")

## BGE-M3

In [ ]:
from tqdm.auto import tqdm

emb_bge = cargar_cache_full("bge_m3")
if emb_bge is None:
    from src.llm_clients import get_embedding_client, load_config

    config = load_config()
    cliente = get_embedding_client(config)
    LOTE = 32
    GUARDA_CADA = 20  # lotes entre checkpoints (~640 artículos)
    ruta_parcial = EMB_DIR / "bge_m3_full_parcial.npy"

    # Reanuda desde el parcial si existe y no lo desborda (si cambiaste el corpus, bórralo).
    vectores = list(np.load(ruta_parcial)) if ruta_parcial.exists() else []
    if len(vectores) >= len(textos):
        vectores = []  # parcial obsoleto/ajeno
    inicio = len(vectores)
    if inicio:
        print(f"Reanudando BGE-M3 desde {inicio}/{len(textos)}")

    pasos = list(range(inicio, len(textos), LOTE))
    for paso, i in enumerate(tqdm(pasos, desc="BGE-M3")):
        respuesta = cliente.embeddings.create(model=config.embedding_model,
                                              input=textos[i:i + LOTE])
        vectores.extend(d.embedding for d in respuesta.data)
        if (paso + 1) % GUARDA_CADA == 0:
            np.save(ruta_parcial, np.array(vectores, dtype=np.float32))  # checkpoint

    emb_bge = np.array(vectores, dtype=np.float32)
    guardar_cache_full("bge_m3", emb_bge)
    ruta_parcial.unlink(missing_ok=True)
print(f"Embeddings BGE-M3: {emb_bge.shape}")

## BETO (chunking + mean pooling)

In [ ]:
# --- BETO por articulo: chunking + mean pooling, reanudable por bloques ---
# Un vector [768] por articulo (media de sus chunks). El chunking solo trocea si el texto
# supera los 512 tokens de BETO (mediana del corpus: 1 chunk). Pooling inter-chunk = media
# (una atencion sin parametros resulto indistinguible, coseno 0.9999). Los parametros por
# defecto estan dimensionados para GPU; en CPU baja batch_chunks a ~64.
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

MODELO_BETO = "dccuchile/bert-base-spanish-wwm-cased"


def partir_en_chunks(tokenizador, texto, max_tokens=512, stride_ratio=0.2):
    ids = tokenizador(texto, add_special_tokens=False, truncation=False)["input_ids"]
    utiles = max_tokens - 2  # espacio para [CLS] y [SEP]
    paso = max(1, int(utiles * (1 - stride_ratio)))
    cls_id, sep_id = tokenizador.cls_token_id, tokenizador.sep_token_id
    chunks = [[cls_id, *ids[i:i + utiles], sep_id] for i in range(0, max(len(ids), 1), paso)]
    if len(chunks) > 1 and len(chunks[-1]) < utiles * stride_ratio:
        chunks.pop()  # ventana final residual por el solape
    return chunks


@torch.inference_mode()
def _codificar_chunks(modelo, chunks, pad_id, device, usar_amp):
    largo = max(len(c) for c in chunks)
    input_ids = torch.full((len(chunks), largo), pad_id, dtype=torch.long)
    mascara = torch.zeros((len(chunks), largo), dtype=torch.long)
    for i, c in enumerate(chunks):
        input_ids[i, :len(c)] = torch.tensor(c)
        mascara[i, :len(c)] = 1
    input_ids = input_ids.to(device, non_blocking=True)
    mascara = mascara.to(device, non_blocking=True)
    with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=usar_amp):
        salida = modelo(input_ids=input_ids, attention_mask=mascara).last_hidden_state
    salida = salida.float()                       # pooling en fp32 aunque el forward sea bf16
    m = mascara.unsqueeze(-1).float()
    return (salida * m).sum(dim=1) / m.sum(dim=1).clamp(min=1e-9)  # mean pooling intra-chunk


@torch.inference_mode()
def embed_beto_articulos(textos, nombre_cache, ids_cache, max_tokens=512, stride_ratio=0.2,
                         batch_chunks=512, bloque_articulos=2048, guardar_cada=4,
                         device=None, normalizar=True):
    """(len(textos), 768). Checkpoint cada `guardar_cada` bloques; reanuda desde el parcial."""
    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    usar_amp = device.type == "cuda"  # bf16 solo en GPU; en CPU degrada velocidad/precision
    print(f"BETO en device: {device}  (autocast bf16: {usar_amp})")
    tok = AutoTokenizer.from_pretrained(MODELO_BETO, use_fast=True)
    modelo = AutoModel.from_pretrained(MODELO_BETO, use_safetensors=True).to(device).eval()

    ruta_parcial = EMB_DIR / f"{nombre_cache}_full_parcial.npy"
    salida = list(np.load(ruta_parcial)) if ruta_parcial.exists() else []
    if len(salida) >= len(textos):
        salida = []  # parcial obsoleto/ajeno (si cambiaste el corpus, borralo tu)
    inicio = len(salida)
    if inicio:
        print(f"Reanudando BETO desde {inicio}/{len(textos)}")

    bloques = list(range(inicio, len(textos), bloque_articulos))
    for n, b0 in enumerate(tqdm(bloques, desc="BETO (bloques)")):
        bloque = textos[b0:b0 + bloque_articulos]
        chunks_por_texto = [partir_en_chunks(tok, t, max_tokens, stride_ratio) for t in bloque]
        plano = [c for chunks in chunks_por_texto for c in chunks]

        # Ordenar por longitud agrupa chunks parecidos en el mismo batch y evita que un
        # chunk corto obligue a padear hasta 512. Se deshace el orden justo despues.
        orden = sorted(range(len(plano)), key=lambda i: len(plano[i]))
        vecs_ord = torch.cat([
            _codificar_chunks(modelo, [plano[j] for j in orden[i:i + batch_chunks]],
                              tok.pad_token_id, device, usar_amp).cpu()
            for i in range(0, len(orden), batch_chunks)
        ])
        vecs = torch.empty_like(vecs_ord)
        vecs[torch.tensor(orden)] = vecs_ord   # restaura el orden original de `plano`

        ini = 0
        for chunks in chunks_por_texto:
            salida.append(vecs[ini:ini + len(chunks)].mean(dim=0).numpy())  # media inter-chunk
            ini += len(chunks)
        if (n + 1) % guardar_cada == 0:
            np.save(ruta_parcial, np.array(salida, dtype=np.float32))  # checkpoint

    emb = torch.tensor(np.array(salida, dtype=np.float32))
    if normalizar:
        emb = F.normalize(emb, p=2, dim=1)  # L2: euclidea monotona con la coseno
    emb = emb.numpy()
    guardar_cache_full(nombre_cache, emb)
    ruta_parcial.unlink(missing_ok=True)
    return emb


emb_beto = cargar_cache_full("beto")
if emb_beto is None:
    emb_beto = embed_beto_articulos(textos, "beto", ids_full)
print(f"Embeddings BETO: {emb_beto.shape}")

## Visualización 3D: PCA y UMAP

Para cada método se proyecta a **3D** de dos formas:

- **PCA** (lineal, preserva estructura global).
- **UMAP** (no lineal, prioriza vecindades locales pero conserva razonablemente la
  estructura global), sobre una reducción previa a 50 componentes — práctica estándar para
  estabilizar y acelerar.

**t-SNE queda fuera a propósito.** Con ~26k documentos y tres métodos, `sklearn.manifold.TSNE`
son decenas de minutos por método (es CPU puro: no se acelera con GPU), y aporta poco sobre
UMAP, que además conserva mejor las distancias entre grupos. Si se quiere recuperar, la vía
razonable es `cuml.TSNE(method='fft')` en GPU, no la implementación de sklearn.

Los puntos se dibujan con `s=2, alpha=0.12`: con 26k puntos en 3D, los tamaños del EDA
producen una masa sólida en la que el último grupo dibujado tapa a todos los demás.


In [ ]:
from sklearn.decomposition import PCA
from umap import UMAP


def proyectar(emb):
    """Devuelve (PCA 3D, UMAP 3D, varianza explicada del PCA 3D)."""
    red3 = PCA(n_components=3, random_state=RANDOM_STATE)
    xyz_pca = red3.fit_transform(emb)
    var3 = red3.explained_variance_ratio_.sum()

    # Reducción previa a 50 componentes: acelera UMAP y filtra direcciones de ruido.
    x50 = PCA(n_components=min(50, emb.shape[1]), random_state=RANDOM_STATE).fit_transform(emb)
    xyz_umap = UMAP(n_components=3, n_neighbors=15, min_dist=0.1,
                    random_state=RANDOM_STATE).fit_transform(x50)
    return xyz_pca, xyz_umap, var3


metodos = {
    "Doc2Vec": emb_d2v,
    "BGE-M3": emb_bge,
    "BETO": emb_beto,
}
proyecciones = {nombre: proyectar(emb) for nombre, emb in metodos.items()}
print(f"Proyecciones calculadas para {len(proyecciones)} métodos.")

In [ ]:
def plot_proyecciones(color_por: str, titulo: str, colores: dict, etiquetas: dict):
    # Una fila por método: el número de filas se deriva de `proyecciones`, nunca fijo,
    # para que añadir un método no lo deje fuera del gráfico en silencio.
    nfilas = len(proyecciones)
    fig, axes = plt.subplots(nfilas, 2, figsize=(13, 5.8 * nfilas), squeeze=False,
                             subplot_kw={"projection": "3d"})
    for fila, (nombre, (xyz_pca, xyz_umap, var3)) in enumerate(proyecciones.items()):
        vistas = [(xyz_pca, f"PCA ({var3:.0%} var.)"), (xyz_umap, "UMAP")]
        for col, (xyz, tipo) in enumerate(vistas):
            ax = axes[fila, col]
            for valor, color in colores.items():
                mascara = (data[color_por] == valor).to_numpy()
                ax.scatter(xyz[mascara, 0], xyz[mascara, 1], xyz[mascara, 2],
                           s=2, alpha=0.12, color=color, label=etiquetas[valor],
                           linewidths=0)
            ax.set_title(f"{nombre} — {tipo}", fontsize=11)
            ax.set_xticklabels([])
            ax.set_yticklabels([])
            ax.set_zticklabels([])
    handles, labels_leg = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels_leg, loc="upper center", ncol=min(len(colores), 6),
               frameon=False, bbox_to_anchor=(0.5, 1.0), markerscale=8)
    fig.suptitle(titulo, y=1.01, fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.99])
    plt.show()


plot_proyecciones("nombre_periodico", "Embeddings coloreados por periódico",
                  dict(zip(orden_per, PALETA)), {p: p for p in orden_per})

In [ ]:
# Colorear por sección canónica: las 8 clases temáticas más frecuentes; el resto va a
# "Otras" en gris de fondo (con 17 colores el gráfico deja de ser legible).
NO_TEMATICAS = {"Portada", "Opinión", "Actualidad", "Otros"}
top_secciones = (data.loc[~data["seccion_canonica"].isin(NO_TEMATICAS), "seccion_canonica"]
                 .value_counts().head(8).index.tolist())
print("Secciones destacadas:", top_secciones)

data["seccion_top"] = data["seccion_canonica"].where(
    data["seccion_canonica"].isin(top_secciones), "Otras")

# "Otras" va primero para quedar al fondo. Paleta Okabe-Ito (accesible para daltonismo).
PALETA_SEC = ["#0072B2", "#D55E00", "#009E73", "#CC79A7",
              "#E69F00", "#56B4E9", "#F0E442", "#000000"]
colores_sec = {"Otras": "#dedcd2", **dict(zip(top_secciones, PALETA_SEC))}

plot_proyecciones("seccion_top",
                  "Embeddings coloreados por sección canónica (top 8 temáticas)",
                  colores_sec, {v: v for v in colores_sec})

In [ ]:
plot_proyecciones("u_index", "Embeddings coloreados por flag de incertidumbre (u_index)",
                  COLOR_FLAG, {0: "sin incertidumbre", 1: "con incertidumbre"})

### Lectura de las proyecciones

- **Doc2Vec** se entrena sobre este mismo corpus, sin conocimiento externo: es el baseline
  distribucional. Con ~26k documentos ya tiene material suficiente (a diferencia de una
  submuestra de 1,500), así que sus agrupaciones son interpretables, pero siguen siendo
  co-ocurrencias de vocabulario, no semántica.
- **BGE-M3** procesa el artículo completo en un solo forward y tiene fine-tuning
  contrastivo: es el candidato natural para el pipeline de RAG con Milvus.
- **BETO** procesa el artículo entero vía chunking, pero es el único **sin fine-tuning
  contrastivo**: su espacio es anisotrópico (cosenos ~0.94 entre documentos arbitrarios),
  así que una nube compacta y sin estructura aparente puede ser geometría del espacio y no
  ausencia de contenido.
- Cuidado con la varianza explicada por el PCA de BETO: un porcentaje **alto** en 3
  componentes no es buena señal, sino el síntoma de esa dirección común dominante.
- Si los colores por periódico se mezclan dentro de los grupos, la estructura dominante es
  **temática** y no de medio — que es lo deseable para clustering de tópicos. Si en cambio
  los grupos coinciden con el periódico, el embedding está capturando estilo editorial.
- La vista por sección canónica es la lectura previa a la sección 4: si ahí ya se distinguen
  bloques de color, es esperable que HDBSCAN los recupere.


---
# 4. Clustering — HDBSCAN

**HDBSCAN** sobre cada método de embedding. Es basado en densidad, así que no exige fijar
`k` y admite que parte del corpus quede **sin clúster** (etiqueta `-1`, «ruido»), lo cual es
realista en prensa: muchos artículos no pertenecen a ningún tema recurrente.

Pipeline por método, idéntico para los tres para que la comparación sea limpia:

1. **L2-normalizar** el embedding ⇒ la distancia euclídea se vuelve monótona con la coseno.
2. **UMAP a 10 dimensiones** (`metric="cosine"`, `min_dist=0.0`). Con cientos de dimensiones
   la densidad se aplana (maldición de la dimensionalidad) y HDBSCAN degenera; `min_dist=0.0`
   favorece clústeres compactos, que es lo que interesa para clusterizar (no para mirar).
3. **HDBSCAN** sobre ese espacio, con métrica euclídea (el coseno ya se aplicó en el paso 1).

Se usa el paquete standalone `hdbscan` y no `sklearn.cluster.HDBSCAN` porque expone
`condensed_tree_`, `single_linkage_tree_` y `validity_index` (DBCV), que se usan abajo.


In [ ]:
# --- HDBSCAN sobre cada metodo de embedding ---
import hdbscan
from sklearn.preprocessing import normalize
from umap import UMAP

MIN_CLUSTER_SIZE = 50   # tamaño mínimo de un cluster "real"
MIN_SAMPLES = 10        # mayor => más puntos marcados como ruido

resultados = {}
for nombre, emb in metodos.items():
    print(f"\n{'=' * 60}\n{nombre} — {emb.shape}\n{'=' * 60}")
    emb_norm = normalize(emb)

    # Reduccion densidad-preservante; el coseno se aplica aqui, no en HDBSCAN.
    X = UMAP(n_components=10, n_neighbors=30, min_dist=0.0, metric="cosine",
             random_state=RANDOM_STATE).fit_transform(emb_norm)

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        min_samples=MIN_SAMPLES,
        metric="euclidean",
        cluster_selection_method="eom",
        gen_min_span_tree=True,
    )
    labels = clusterer.fit_predict(X)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_ruido = int((labels == -1).sum())
    print(f"Clusters: {n_clusters}  |  ruido: {n_ruido:,} ({n_ruido / len(labels):.1%})")
    if n_clusters:
        tam = np.bincount(labels[labels >= 0])
        print(f"Tamaños — mín {tam.min()}, mediana {int(np.median(tam))}, máx {tam.max()}")

    resultados[nombre] = {"labels": labels, "clusterer": clusterer,
                          "X": X, "emb_norm": emb_norm}

In [ ]:
# Árbol condensado de HDBSCAN, la viz canónica del algoritmo: el grosor de cada rama es
# proporcional al nº de puntos y las elipses marcan los clusters seleccionados. No se
# reajusta nada: el clusterer se creó con gen_min_span_tree=True.
for nombre, r in resultados.items():
    n_cl = len(set(r["labels"]) - {-1})
    if n_cl < 1:
        print(f"{nombre}: sin clusters, se omite el árbol")
        continue
    fig, ax = plt.subplots(figsize=(14, 6))
    r["clusterer"].condensed_tree_.plot(
        select_clusters=True,
        selection_palette=sns.color_palette("tab20", n_cl),
        axis=ax,
    )
    ax.set_title(f"{nombre} — árbol condensado (λ = 1/distancia; ramas ∝ nº de puntos)",
                 fontsize=12)
    plt.tight_layout()
    plt.show()

## Métricas de clustering

Tres familias, porque ninguna basta sola:

- **Descriptivas** — nº de clústeres, % de ruido y tamaños. Un método que manda el 60% del
  corpus a ruido puede sacar buenas métricas sobre el 40% restante; el % de ruido tiene que
  leerse **junto** a todo lo demás.
- **Internas** (sin usar labels) — silhouette y Davies-Bouldin, calculados **excluyendo el
  ruido**, en dos espacios:
  - sobre el **UMAP-10** en el que se clusterizó, y
  - sobre el **embedding original** con métrica coseno.

  Las dos, porque el silhouette sobre UMAP está **inflado por construcción** (UMAP separa
  grupos aunque la separación no exista en el espacio original) y solo el segundo es
  comparable entre métodos de distinta dimensión. Se añade **DBCV**
  (`hdbscan.validity_index`), la métrica interna diseñada para clústeres de densidad: a
  diferencia del silhouette, no penaliza formas alargadas o no convexas.
- **Externas** contra `seccion_canonica` — ARI, AMI, homogeneidad, completitud, V-measure y
  pureza media. Aquí el ruido **sí se cuenta**, como una categoría más: ignorarlo premiaría
  a los métodos que descartan lo difícil.

Las externas se reportan en **dos vistas**: sobre las 17 clases, y sobre las 13 temáticas
(excluyendo `Portada`, `Opinión`, `Actualidad` y `Otros`, que no son temas — ver sección 2).

Silhouette y DBCV se calculan sobre una **submuestra** (`N_MUESTRA_METRICA`) porque ambos
necesitan la matriz de distancias completa: con 26k puntos serían ~5 GB.


In [ ]:
from sklearn.metrics import (adjusted_mutual_info_score, adjusted_rand_score,
                             davies_bouldin_score,
                             homogeneity_completeness_v_measure, silhouette_score)

N_MUESTRA_METRICA = 8000   # silhouette/DBCV necesitan la matriz de distancias completa

verdad = data["seccion_canonica"].to_numpy()   # NO_TEMATICAS viene de la sección 3
mask_tematica = ~np.isin(verdad, list(NO_TEMATICAS))
rng = np.random.default_rng(RANDOM_STATE)


def _internas(X, emb_norm, labels):
    """Silhouette/DB sobre el espacio de clustering y sobre el embedding original."""
    sin_ruido = labels != -1
    if sin_ruido.sum() < 2 or len(set(labels[sin_ruido])) < 2:
        return {}
    idx = np.flatnonzero(sin_ruido)
    if len(idx) > N_MUESTRA_METRICA:
        idx = rng.choice(idx, N_MUESTRA_METRICA, replace=False)
    y = labels[idx]
    if len(set(y)) < 2:
        return {}
    return {
        "silhouette_umap": silhouette_score(X[idx], y),
        "silhouette_cos": silhouette_score(emb_norm[idx], y, metric="cosine"),
        "davies_bouldin": davies_bouldin_score(X[idx], y),
    }


def _dbcv(X, labels):
    """DBCV (validity_index) sobre submuestra; requiere float64 y matriz de distancias."""
    idx = np.arange(len(labels))
    if len(idx) > N_MUESTRA_METRICA:
        idx = rng.choice(idx, N_MUESTRA_METRICA, replace=False)
    y = labels[idx]
    if len(set(y) - {-1}) < 2:
        return np.nan
    try:
        return hdbscan.validity.validity_index(X[idx].astype(np.float64), y)
    except Exception as e:                      # DBCV falla si un cluster queda con 1 punto
        print(f"  (DBCV no calculable: {e})")
        return np.nan


def _externas(labels, verdad, mascara, sufijo):
    """ARI/AMI/homogeneidad/completitud/V y pureza. El ruido cuenta como categoría."""
    y_pred, y_true = labels[mascara], verdad[mascara]
    h, c, v = homogeneity_completeness_v_measure(y_true, y_pred)
    # Pureza: fracción de cada cluster que cae en su clase mayoritaria, ponderada por tamaño.
    pureza = sum(pd.Series(y_true[y_pred == cl]).value_counts().iloc[0]
                 for cl in set(y_pred)) / len(y_pred)
    return {f"ARI{sufijo}": adjusted_rand_score(y_true, y_pred),
            f"AMI{sufijo}": adjusted_mutual_info_score(y_true, y_pred),
            f"homog{sufijo}": h, f"compl{sufijo}": c, f"V{sufijo}": v,
            f"pureza{sufijo}": pureza}


filas = []
for nombre, r in resultados.items():
    labels = r["labels"]
    n_ruido = int((labels == -1).sum())
    tam = np.bincount(labels[labels >= 0]) if (labels >= 0).any() else np.array([0])
    fila = {
        "metodo": nombre,
        "dims": metodos[nombre].shape[1],
        "n_clusters": len(set(labels)) - (1 if -1 in labels else 0),
        "%_ruido": 100 * n_ruido / len(labels),
        "tam_min": int(tam.min()), "tam_med": int(np.median(tam)), "tam_max": int(tam.max()),
    }
    fila.update(_internas(r["X"], r["emb_norm"], labels))
    fila["DBCV"] = _dbcv(r["X"], labels)
    fila.update(_externas(labels, verdad, np.ones(len(labels), bool), "_17"))
    fila.update(_externas(labels, verdad, mask_tematica, "_13"))
    filas.append(fila)

metricas = pd.DataFrame(filas).set_index("metodo")
print("Descriptivas e internas (silhouette/DBCV: ↑ mejor; Davies-Bouldin: ↓ mejor)")
display(metricas[["dims", "n_clusters", "%_ruido", "tam_min", "tam_med", "tam_max",
                  "silhouette_umap", "silhouette_cos", "davies_bouldin", "DBCV"]].round(3))

print("\nExternas vs sección canónica — _17: las 17 clases | _13: solo temáticas (↑ mejor)")
display(metricas[[c for c in metricas.columns
                  if c.endswith("_17") or c.endswith("_13")]].round(3))

In [ ]:
# Matriz cluster x sección canónica del mejor método según AMI sobre las clases temáticas.
# Normalizada por fila: cada celda es el % del cluster que cae en esa sección, así que una
# fila concentrada en una columna significa un cluster temáticamente puro.
MEJOR = metricas["AMI_13"].idxmax()
print(f"Mejor método por AMI_13: {MEJOR}")

labels_mejor = resultados[MEJOR]["labels"]
cruce = pd.crosstab(pd.Series(labels_mejor, name="cluster"),
                    pd.Series(verdad, name="seccion_canonica"), normalize="index") * 100

fig, ax = plt.subplots(figsize=(13, max(4, 0.32 * len(cruce))))
sns.heatmap(cruce, cmap="Blues", vmin=0, vmax=100, linewidths=0.5, linecolor="white",
            cbar_kws={"label": "% del cluster"}, ax=ax)
ax.set_title(f"{MEJOR} — composición de cada cluster por sección canónica "
             f"(fila -1 = ruido)", fontsize=12)
plt.tight_layout()
plt.show()

### Nombrar los clústeres con un LLM (opcional)

Pide a DeepSeek un título descriptivo por clúster a partir de una muestra de sus artículos.
**Requiere VPN GlobalProtect activa** y consume tokens del servidor de la universidad, así
que se ejecuta para un solo método (`METODO_TITULAR`), no para los tres.


In [ ]:
# %% Nombrar cada cluster con DeepSeek (mismo patrón que hc_clustering.ipynb)
from src.llm_clients import get_chat_client, load_config
from tqdm.auto import tqdm

METODO_TITULAR = MEJOR          # cámbialo si quieres titular otro método
labels_tit = resultados[METODO_TITULAR]["labels"]

config = load_config()
cliente_chat = get_chat_client(config)
N_EJEMPLOS = 10

PROMPT_TITULO = (
    "Eres un analista de prensa ecuatoriana. A continuación tienes una muestra de "
    "artículos (título y primeras líneas) que pertenecen a un mismo clúster temático "
    "obtenido por clustering no supervisado.\n\n{ejemplos}\n\n"
    "Escribe UN solo título descriptivo en español (máximo 10 palabras) que resuma el "
    "tema dominante del clúster. Si los temas son heterogéneos, dilo en el título "
    "(p. ej. 'Actualidad general: política, deportes y sucesos'). "
    "Responde únicamente con el título, sin comillas ni explicaciones."
)


def titular_cluster(indices: np.ndarray) -> str:
    """Pide a DeepSeek un título descriptivo para los artículos indicados (posiciones en `data`)."""
    ejemplos = "\n".join(
        f"- {data.iloc[i]['titulo'] or '(sin título)'}: {data.iloc[i]['texto_norm'][:200]}…"
        for i in indices
    )
    respuesta = cliente_chat.chat.completions.create(
        model=config.chat_model,
        messages=[{"role": "user", "content": PROMPT_TITULO.format(ejemplos=ejemplos)}],
        temperature=0.3,
        max_tokens=2000,  # modelo de razonamiento: reserva presupuesto para 'reasoning'
    )
    contenido = respuesta.choices[0].message.content
    return contenido.strip().strip('"') if contenido else "(respuesta vacía)"


rng_tit = np.random.default_rng(RANDOM_STATE)
filas = []
for c in tqdm(sorted(set(labels_tit) - {-1}), desc=f"Titulando ({METODO_TITULAR})"):
    indices = np.flatnonzero(labels_tit == c)
    seleccion = rng_tit.choice(indices, size=min(N_EJEMPLOS, len(indices)), replace=False)
    filas.append({
        "cluster": int(c),
        "n_articulos": len(indices),
        "seccion_dominante": pd.Series(verdad[indices]).value_counts().index[0],
        "titulo_deepseek": titular_cluster(seleccion),
    })

titulos_clusters = pd.DataFrame(filas)
# Mapa cluster -> título, con etiqueta explícita para el ruido
nombre_cluster = dict(zip(titulos_clusters["cluster"], titulos_clusters["titulo_deepseek"]))
nombre_cluster[-1] = "Ruido (sin clúster)"
with pd.option_context("display.max_colwidth", 120):
    display(titulos_clusters.sort_values("n_articulos", ascending=False))

In [ ]:
# %% Graficar los clusters en 2D (UMAP) con los nombres de DeepSeek
# Proyección 2D solo para la viz, independiente del UMAP-10 usado para clusterizar.
xy = UMAP(n_components=2, n_neighbors=30, min_dist=0.1, metric="cosine",
          random_state=RANDOM_STATE).fit_transform(resultados[METODO_TITULAR]["emb_norm"])

fig, ax = plt.subplots(figsize=(13, 10))
clusters_validos = sorted(set(labels_tit) - {-1})
cmap = plt.get_cmap("tab20", max(len(clusters_validos), 1))

# Ruido primero, en gris de fondo
mask_ruido = labels_tit == -1
ax.scatter(xy[mask_ruido, 0], xy[mask_ruido, 1], s=3, c="#c3c2b7",
           alpha=0.25, linewidths=0, label="_nolegend_")

for j, c in enumerate(clusters_validos):
    m = labels_tit == c
    color = cmap(j)
    ax.scatter(xy[m, 0], xy[m, 1], s=5, color=color, alpha=0.5, linewidths=0)
    # Etiqueta el cluster en su centroide (título recortado para que quepa)
    cx, cy = xy[m, 0].mean(), xy[m, 1].mean()
    etiqueta = nombre_cluster.get(c, str(c))
    ax.annotate(f"{c}: {etiqueta[:38]}", (cx, cy), fontsize=8, fontweight="bold",
                ha="center", va="center", color="#2b2b2b",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=color, alpha=0.85))

ax.set_xticks([]); ax.set_yticks([])
n_ruido = int(mask_ruido.sum())
ax.set_title(f"{METODO_TITULAR} — HDBSCAN sobre {len(labels_tit):,} artículos: "
             f"{len(clusters_validos)} clústeres + {n_ruido:,} en ruido "
             f"({n_ruido / len(labels_tit):.0%})", fontsize=12)
plt.tight_layout()
plt.show()

---
# 5. AE / VAE — reducción de dimensión (exploratorio)

Autoencoder / VAE puramente feed-forward (sin secuencia ni attention pooling): recibe
**cualquier** matriz de embeddings `[N, D]` y comprime a `latent_dim`, para aplicarse por
igual a Doc2Vec, BGE-M3 y BETO. El AE aprende una compresión determinista; el VAE una
distribución latente regularizada con KL (se usa `mu` para clusterizar, más estable que una
muestra). Luego se clusteriza el latente con GMM/KMeans.

**Sección exploratoria, sin integrar en el pipeline:** no alimenta las métricas de la
sección 4 ni se ha decidido si formará parte del trabajo final.


In [ ]:
# --- AE / VAE (MLP) generico: opera sobre cualquier embedding [N, D] ---
import lightning as L
import torch
import torch.nn as nn
import torch.nn.functional as F
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.preprocessing import StandardScaler


class MLPAutoencoder(L.LightningModule):
    """AE (variational=False) o VAE (True) feed-forward. hidden_dims define el encoder;
    el decoder es su espejo. Reconstruye el vector de entrada (no una secuencia)."""

    def __init__(self, input_dim, hidden_dims=(256,), latent_dim=64, variational=False,
                 dropout=0.0, lr=1e-3, weight_decay=1e-5,
                 beta=0.1, beta_max=1.0, beta_warmup_steps=0):
        super().__init__()
        self.save_hyperparameters()

        def bloque(i, o):
            return [nn.Linear(i, o), nn.GELU(), nn.Dropout(dropout)]

        enc_dims = [input_dim, *hidden_dims]
        self.encoder = nn.Sequential(
            *[capa for i, o in zip(enc_dims[:-1], enc_dims[1:]) for capa in bloque(i, o)]
        )
        ultimo = enc_dims[-1]
        if variational:
            self.to_mu = nn.Linear(ultimo, latent_dim)
            self.to_logvar = nn.Linear(ultimo, latent_dim)
        else:
            self.to_latent = nn.Linear(ultimo, latent_dim)

        dec_dims = [latent_dim, *reversed(hidden_dims)]
        capas = [c for i, o in zip(dec_dims[:-1], dec_dims[1:]) for c in bloque(i, o)]
        capas.append(nn.Linear(dec_dims[-1], input_dim))
        self.decoder = nn.Sequential(*capas)

    def _encode(self, x):
        h = self.encoder(x)
        if self.hparams.variational:
            mu, logvar = self.to_mu(h), self.to_logvar(h)
            z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)  # reparametrizacion
            return z, mu, logvar
        return self.to_latent(h), None, None

    def forward(self, x):
        z, mu, logvar = self._encode(x)
        return self.decoder(z), z, mu, logvar

    def _beta(self):
        if not self.hparams.variational:
            return 0.0
        if self.hparams.beta_warmup_steps <= 0:
            return self.hparams.beta
        p = min(1.0, self.global_step / float(self.hparams.beta_warmup_steps))
        return self.hparams.beta + p * (self.hparams.beta_max - self.hparams.beta)  # KL annealing

    def _step(self, batch, stage):
        x = batch[0]
        recon, z, mu, logvar = self(x)
        recon_loss = F.mse_loss(recon, x)
        if self.hparams.variational:
            kl = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
            loss = recon_loss + self._beta() * kl
            self.log(f"{stage}_kl", kl, on_step=False, on_epoch=True)
        else:
            loss = recon_loss
        self.log(f"{stage}_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._step(batch, "train")

    def validation_step(self, batch, batch_idx):
        self._step(batch, "val")

    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        _, z, mu, _ = self(batch[0])
        return (mu if self.hparams.variational else z).detach().cpu()  # mu: mas estable para clustering

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr,
                                weight_decay=self.hparams.weight_decay)
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3)
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sch, "monitor": "val_loss"}}


def entrenar_autoencoder(emb, variational=False, latent_dim=64, hidden_dims=(256,),
                         max_epochs=50, batch_size=128, lr=1e-3, beta=0.05,
                         beta_warmup_steps=500, estandarizar=True, seed=42):
    """Entrena un AE/VAE MLP sobre `emb` [N, D] y devuelve los latentes [N, latent_dim] en
    el MISMO orden que `emb` (latentes[i] <-> emb[i]), listos para GMM/KMeans."""
    X = np.asarray(emb, dtype=np.float32)
    if estandarizar:
        X = StandardScaler().fit_transform(X).astype(np.float32)  # cada metodo tiene su escala
    X = torch.from_numpy(X)

    ds = TensorDataset(X)
    n_val = max(1, int(0.1 * len(ds)))
    tr, va = random_split(ds, [len(ds) - n_val, n_val],
                          generator=torch.Generator().manual_seed(seed))
    dl_tr = DataLoader(tr, batch_size=batch_size, shuffle=True)
    dl_va = DataLoader(va, batch_size=batch_size)
    dl_full = DataLoader(ds, batch_size=batch_size)  # shuffle=False -> conserva el orden

    modelo = MLPAutoencoder(input_dim=X.shape[1], hidden_dims=hidden_dims, latent_dim=latent_dim,
                            variational=variational, lr=lr, beta=beta,
                            beta_warmup_steps=beta_warmup_steps if variational else 0)
    trainer = Trainer(
        max_epochs=max_epochs, accelerator="auto", devices=1,  # 1 proceso: DDP no corre en notebook y fragmentaria predict
        precision="16-mixed" if torch.cuda.is_available() else "32-true",
        logger=False, enable_checkpointing=False,
        callbacks=[EarlyStopping(monitor="val_loss", patience=8, mode="min")],
    )
    trainer.fit(modelo, dl_tr, dl_va)
    return torch.cat(trainer.predict(modelo, dl_full), dim=0).numpy()

In [ ]:
# Aplicar a cualquiera de los metodos (cada emb_* es [N, D]). Ajusta latent_dim/hidden_dims/beta por metodo.
lat_beto_ae  = entrenar_autoencoder(emb_beto, variational=False)
lat_beto_vae = entrenar_autoencoder(emb_beto, variational=True, beta=0.05)
print("BETO  AE:", lat_beto_ae.shape, " VAE:", lat_beto_vae.shape)

# Cuando tengas emb_bge / emb_d2v para TODOS los textos, mismo patron:
# lat_bge_vae = entrenar_autoencoder(emb_bge, variational=True)
# lat_d2v_ae  = entrenar_autoencoder(emb_d2v, variational=False)

# Clustering sobre el latente (ejemplo):
# from sklearn.mixture import GaussianMixture
# labels = GaussianMixture(n_components=10, covariance_type="full",
#                          random_state=42).fit_predict(lat_beto_vae)